# Control Variates in QMC.jl

This notebook mirrors the QMCPy `control_variates.ipynb` demo while using Julia's `QMC.jl` APIs. It demonstrates current support for linear control variates in both Monte Carlo and quasi-Monte Carlo stopping criteria.

## Setup

In [1]:
using QMC
using Printf
using Statistics

sample_count(result) = haskey(result.data, :n_total) ? result.data[:n_total] : result.data[:n]

function compare(problem, discrete_distrib, stopping_crit; abs_tol)
    g, cvs, cvmus = problem(discrete_distrib)
    sc = stopping_crit(g; abs_tol = abs_tol)
    result = integrate(sc)
    sc_cv = stopping_crit(
        g;
        abs_tol = abs_tol,
        control_variates = cvs,
        control_variate_means = cvmus,
    )
    result_cv = integrate(sc_cv)
    name = nameof(typeof(sc))
    @printf("Stopping Criterion: %-15s absolute tolerance: %.1e\n", String(name), abs_tol)
    @printf("  Without CV: solution %-10.5f samples %.1e\n", result.solution, float(sample_count(result)))
    @printf("  With CV:    solution %-10.5f samples %.1e\n", result_cv.solution, float(sample_count(result_cv)))
    @printf("  Sample ratio with CV: %.1f%%\n", 100 * sample_count(result_cv) / sample_count(result))
    println("  Control-variate beta: ", result_cv.data[:control_variate_beta])
    println()
    return result, result_cv
end


compare (generic function with 1 method)

## Problem 1: Polynomial Function

We will integrate

`g(t) = 10 t_1 - 5 t_2^2 + 2 t_3^3`

with true measure `U[0,2]^3` and control variates

`g_hat_1(t) = t_1`

and

`g_hat_2(t) = t_2^2`

using the same true measure.

In [2]:
function poly_problem(dd)
    tm = Uniform(dd; lower_bound = 0.0, upper_bound = 2.0)
    g = CustomFun(tm, x -> 10 .* x[:, 1] .- 5 .* x[:, 2].^2 .+ 2 .* x[:, 3].^3)
    cv1 = CustomFun(tm, x -> x[:, 1])
    cv2 = CustomFun(tm, x -> x[:, 2].^2)
    return g, [cv1, cv2], [1.0, 4 / 3]
end

compare(poly_problem, IIDStdUniform(3; seed = 7), CubMCCLT; abs_tol = 1e-2)
compare(
    poly_problem,
    DigitalNetB2(3; seed = 7, randomize = "LMS_DS", graycode = false),
    CubQMCNetG;
    abs_tol = 1e-8,
)


┌ Warning: CubMCCLT: did not converge within n_max=1073741824. Error bound: 0.010165840234302458, tolerance: 0.01
└ @ QMC ~/Documents/ProgramData/QMCSoftware_space/QMC.jl/src/stopping_criterion/cub_mc_clt.jl:134
┌ Warning: CubMCCLT: did not converge within n_max=1073741824. Error bound: 0.01008679060691642, tolerance: 0.01
└ @ QMC ~/Documents/ProgramData/QMCSoftware_space/QMC.jl/src/stopping_criterion/cub_mc_clt.jl:134


Stopping Criterion: CubMCCLT        absolute tolerance: 1.0e-02
  Without CV: solution 7.33600    samples 8.3e+06
  With CV:    solution 7.33266    samples 1.9e+06
  Sample ratio with CV: 23.3%
  Control-variate beta: [10.064003732200382, -4.989575962368172]

Stopping Criterion: CubQMCNetG      absolute tolerance: 1.0e-08
  Without CV: solution 7.33334    samples 1.0e+06
  With CV:    solution 7.33334    samples 5.2e+05
  Sample ratio with CV: 50.0%
  Control-variate beta: [9.99098836515091, -4.9996988482162115]



(QMCResult(solution=7.333339e+00, n_total=1048576, error_bound=8.73e-09), QMCResult(solution=7.333345e+00, n_total=524288, error_bound=7.17e-09))

## Problem 2: Keister Function

This problem integrates the Keister function while using control variates

`g_1(x) = sin(pi x)`

and

`g_2(x) = -3 (x - 1/2)^2 + 1`.

As in the QMCPy demo, this Julia version uses the one-dimensional case for readability, but the control-variate API is compatible with higher dimensions.

In [3]:
function keister_problem(dd)
    main = Keister(Gaussian(dd; covariance = 0.5))
    tm = Uniform(dd)
    cv1 = CustomFun(tm, x -> sin.(pi .* x[:, 1]))
    cv2 = CustomFun(tm, x -> -3 .* (x[:, 1] .- 0.5).^2 .+ 1.0)
    return main, [cv1, cv2], [2 / pi, 3 / 4]
end

compare(keister_problem, IIDStdUniform(1; seed = 7), CubMCCLT; abs_tol = 5e-4)
compare(
    keister_problem,
    DigitalNetB2(1; seed = 7, randomize = "LMS_DS", graycode = false),
    CubQMCNetG;
    abs_tol = 1e-7,
)


Stopping Criterion: CubMCCLT        absolute tolerance: 5.0e-04
  Without CV: solution 1.38045    samples 9.4e+06
  With CV:    solution 1.38068    samples 4.4e+05
  Sample ratio with CV: 4.7%
  Control-variate beta: [-8.780962378735063, 14.112709269234074]

Stopping Criterion: CubQMCNetG      absolute tolerance: 1.0e-07
  Without CV: solution 1.38039    samples 2.1e+06
  With CV:    solution 1.38039    samples 1.0e+06
  Sample ratio with CV: 50.0%
  Control-variate beta: [-40.106792275646555, 57.29018092748511]



(QMCResult(solution=1.380388e+00, n_total=2097152, error_bound=5.93e-08), QMCResult(solution=1.380388e+00, n_total=1048576, error_bound=7.55e-08))

## Problem 3: Option Pricing

We use a European call option as a control variate for pricing an arithmetic Asian call option, following the same structure as the QMCPy demo.

In [4]:
call_put = :call
start_price = 100.0
strike_price = 125.0
volatility = 0.75
interest_rate = 0.01
t_final = 1.0
dimension = 12

function option_problem(dd)
    tm = BrownianMotion(dd; t_final = t_final)
    european_cv = FinancialOption(
        tm;
        option_type = :european,
        call_put = call_put,
        volatility = volatility,
        start_price = start_price,
        strike_price = strike_price,
        interest_rate = interest_rate,
    )
    asian_call = FinancialOption(
        tm;
        option_type = :asian,
        call_put = call_put,
        mean_type = :arithmetic,
        volatility = volatility,
        start_price = start_price,
        strike_price = strike_price,
        interest_rate = interest_rate,
    )
    return asian_call, european_cv, get_exact_value(european_cv)
end

compare(option_problem, IIDStdUniform(dimension; seed = 7), CubMCCLT; abs_tol = 5e-2)
compare(
    option_problem,
    DigitalNetB2(dimension; seed = 7, randomize = "LMS_DS", graycode = false),
    CubQMCNetG;
    abs_tol = 1e-3,
)


┌ Warning: CubMCCLT: did not converge within n_max=1073741824. Error bound: 0.05773562942255963, tolerance: 0.05
└ @ QMC ~/Documents/ProgramData/QMCSoftware_space/QMC.jl/src/stopping_criterion/cub_mc_clt.jl:134


Stopping Criterion: CubMCCLT        absolute tolerance: 5.0e-02
  Without CV: solution 10.62464   samples 5.3e+06
  With CV:    solution 10.60631   samples 8.0e+05
  Sample ratio with CV: 15.1%
  Control-variate beta: [0.38944136840048105]

Stopping Criterion: CubQMCNetG      absolute tolerance: 1.0e-03
  Without CV: solution 10.61365   samples 5.2e+05
  With CV:    solution 10.61406   samples 5.2e+05
  Sample ratio with CV: 100.0%
  Control-variate beta: [0.036738117591079526]



(QMCResult(solution=1.061365e+01, n_total=524288, error_bound=8.08e-04), QMCResult(solution=1.061406e+01, n_total=524288, error_bound=8.40e-04))